# 第24章　推論パイプラインを組む ― 入力から結果出力まで

**『医療診断支援AI開発　基礎編 ― 自分で作る（基礎編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-basic

## 推論に要るのは、重みだけではない

In [ ]:
import torch

# 例：第13章の2クラス画像分類（正常／DR）。セグメンテーションの成果物は 24.2 で別に扱う
torch.save({
    "model":        model.state_dict(),        # 重みとバイアス
    "arch":         "efficientnet_b0",         # どのネットワーク定義で読むか
    "task":         "classification",          # 分類か、セグメンテーションか
    "classes":      train_ds.classes,          # 番号→クラス名。これが無いと出力を読めない
    "positive_class": train_ds.class_to_idx["DR"],   # 閾値を掛ける「陽性」の番号
    "img_size":     224,                       # 入力サイズ
    "normalize":    {"mean": [0.485, 0.456, 0.406],
                     "std":  [0.229, 0.224, 0.225]},
    "preprocess":   "val-v2",                  # 前処理レシピの版（コード側と対応づける）
    "threshold":    0.42,                      # 運用点。開発用の検証データで決めた値
    "dataset_version": version,                # どのデータで学習したか（第20章）
    "split_version": split_version,            # どう分けたか（第20章）
}, "model_for_inference.pth")

## 並べて見る ― 同じモデル、違う周辺

In [ ]:
from torch.amp import autocast

# ---- 重みを更新するとき（2クラス画像分類）----
model.train()                                # Dropoutを効かせ、BatchNormは今のバッチで統計を取る
for x, y in train_loader:                    # 正解 y が要る。shuffle=True で順番も混ぜる
    x, y = x.to(device), y.to(device)
    # 拡張は Dataset の transform で掛ける（分類なら画像だけ。セグメンテーションなら
    # 画像とマスクへ同じ幾何変換を掛ける。第25章参照）
    optimizer.zero_grad(set_to_none=True)
    with autocast(dev, enabled=amp_on):
        loss = criterion(model(x), y)        # 損失関数で「どれだけ外したか」を測り
    scaler.scale(loss).backward()            # 勾配を求め
    scaler.step(optimizer); scaler.update()  # 重みとバイアスを更新する

In [ ]:
import torch

# ---- 本番で推論するとき（同じ2クラス画像分類）----
ck = torch.load("model_for_inference.pth", map_location=device, weights_only=False)
assert ck["task"] == "classification", "この推論コードは分類の成果物専用"
model = build_model(ck["arch"], num_classes=len(ck["classes"])).to(device)
model.load_state_dict(ck["model"])
model.eval()                                 # Dropoutを止め、BatchNormは貯めた統計を使う
POS = ck["positive_class"]                   # 陽性クラスの番号は成果物から読む

@torch.no_grad()                             # 勾配を作らない
def predict(image_path):
    x = val_transform(image_path, ck)        # 前処理は「検証用」と完全に同じもの（ckの設定で作る）
    logits = model(x.unsqueeze(0).to(device))          # (1, C)
    prob = logits.softmax(1)[0]                        # (C,)   正解も損失も更新も、ここには無い
    p_pos = float(prob[POS])                           # 陽性クラスの確率
    return {"positive": p_pos >= ck["threshold"],      # 判定は保存した閾値で。argmaxではない
            "p_positive": p_pos,
            "class_if_argmax": ck["classes"][int(prob.argmax())]}   # 参考：最大クラス

## 24.2　推論の全体像

In [ ]:
import torch
from monai.inferers import sliding_window_inference

class InferencePipeline:
    def __init__(self, ckpt_path, device=None):
        device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.device = device
        ck = torch.load(ckpt_path, map_location=device, weights_only=False)
        assert ck["task"] == "segmentation", "このパイプラインはセグメンテーションの成果物専用"
        self.cfg = ck                                       # 閾値・後処理・spacing は全部ここから読む
        self.model = build_model(ck["arch"]).to(device)
        self.model.load_state_dict(ck["model"])
        self.model.eval()                                   # 推論モード
        self.target_spacing = tuple(ck["target_spacing"])   # 例: (1.5, 1.5, 1.5)
        self.pre = build_val_transform(pixdim=self.target_spacing)   # 前処理も成果物の設定で作る

    @torch.no_grad()                                        # 勾配を追跡しない
    def run(self, image_path):
        import nibabel as nib
        src = nib.load(image_path)                          # 元の幾何を、前処理の前に確保する
        data = self.pre({"image": image_path})              # ①前処理
        data["src_affine"] = src.affine                     # 元画像のaffine
        data["src_shape"]  = src.shape                      # 元画像の形
        # MONAIのMetaTensorは、掛かった変換の履歴を applied_operations に持つ。
        # これが逆変換（24.5）の材料になる。独自の辞書キーが自動で生まれるわけではない。
        data["transforms"] = list(data["image"].applied_operations)
        img = data["image"].unsqueeze(0).to(self.device)
        logits = sliding_window_inference(                  # ②スライディングウィンドウ
            img, roi_size=(96, 96, 96), sw_batch_size=2, predictor=self.model, overlap=0.5)
        mask = logits.argmax(1)[0].cpu().numpy()            # ③各ボクセルをクラスへ
        # ④後処理 → ⑤確信度と棄却（24.4）。閾値は成果物から読む
        result = self.postprocess(mask, data)
        return add_confidence_flag(logits, result, thr=self.cfg["threshold"])

## 24.3　後処理と定量化

In [ ]:
import numpy as np

class InferencePipeline:
    # ...（前節の __init__・run はそのまま。このクラスに postprocess を足す）

    def postprocess(self, mask, data):
        import cc3d, numpy as np   # pip install connected-components-3d（配布名に注意）
        # 前処理でリサンプリングした後の spacing を使う。
        # meta["pixdim"] は元のNIfTIヘッダ由来で、リサンプリングしても更新されない。
        spacing = self.target_spacing                        # 例: (1.5, 1.5, 1.5)
        vox_vol = float(np.prod(spacing))
        min_vol = self.cfg["postprocess"]["min_volume_mm3"]  # 直書きしない。成果物の設定を使う
        comps = cc3d.connected_components(mask == 1)         # 病変を塊に分ける
        lesions, kept = [], np.zeros_like(mask)              # kept＝除去後のマスク
        for c in range(1, comps.max() + 1):
            sel = (comps == c)
            n = sel.sum()
            if n * vox_vol < min_vol:                       # 例：30mm³ は等価球径で約3.9mm。対象疾患ごとに必ず見直す
                # （微小結節や微小出血では、この値は大きすぎて真の病変を消してしまう）
                continue
            kept[sel] = mask[sel]                            # 残す塊だけを新しいマスクへ写す
            equiv_d = 2 * ((3*n*vox_vol/(4*np.pi))**(1/3))   # 等価球の径(mm)。体積から直接求め、非等方spacingでも正しい
            lesions.append({"volume_mm3": n*vox_vol,
                            "equivalent_sphere_diameter_mm": equiv_d})  # 臨床の長径とは別物
        # 除去前と除去後を分けて返す。集計・画面表示・DICOM出力はすべて filtered_mask を使う。
        # 一方だけを書き換えると、n_lesions=0 なのに画面には病変が残る、という食い違いが起きる。
        #
        # そしてもう一つ。ここで返すマスクは「前処理後（向き・spacing・cropを変えた後）の
        # 座標系」に乗っている。元画像へ重ねたりDICOMで返したりする前に、必ず元のグリッドへ
        # 戻すこと。そのために、変換の履歴と元の幾何を一緒に持ち帰る。
        return {"raw_mask": mask, "filtered_mask": kept,
                "lesions": lesions, "n_lesions": len(lesions),
                "space": "preprocessed",              # まだ元画像の座標ではない
                "transforms": data["transforms"],     # 逆変換に使う履歴
                "src_affine": data["src_affine"],     # 元画像の幾何
                "src_shape": data["src_shape"]}

## 24.4　確信度と、棄却の判断

In [ ]:
def add_confidence_flag(logits, result, thr):
    """logits: モデル出力 (1, C, D, H, W)
    result: postprocess が返した辞書。filtered_mask（後処理で残った病変）を使う。
    thr: 成果物に書いた運用の閾値（例 0.85）。2クラスでは確信度の下限が0.5なので、
    0.5 だと決して棄権しない。値は開発用の検証データで決める。"""
    fg = logits.softmax(1)[0, 1:].max(0).values          # 前景クラスの確信度マップ (D,H,W)
    # 除去前の raw_mask ではなく、後処理で残った filtered_mask を見る。
    # 後処理で全病変が消えた症例に「病変の確信度」を付けてはいけない。
    m = torch.from_numpy(result["filtered_mask"] > 0).to(fg.device)
    # ※全ボクセル・全クラスの最大値は常に高く出るので、棄却の指標にはならない。
    #   「病変と判定した場所で、どれだけ自信があったか」を代表値にする。
    if m.any():
        conf = fg[m].mean().item()                        # 丸める前の値で判定する
        result["lesion_confidence"] = conf                # 表示するときに丸める（判定には使わない）
        result["flag"] = "要専門医確認" if conf < thr else "通常"   # フラグは必ず明示的に置く
    else:
        # 病変が残らなかった場合。ここを高い確信度で埋めてはいけない。
        # 「病変が無い」と「見つけられなかった（見逃し・対象外入力）」は区別できない。
        result["lesion_confidence"] = None
        result["flag"] = "陰性 ― 陰性の確からしさは本指標では担保されない"
    # 除去前の候補（raw_mask）についての統計は、別の名前で返す
    raw = torch.from_numpy(result["raw_mask"] > 0).to(fg.device)
    result["raw_candidate_confidence"] = fg[raw].mean().item() if raw.any() else None
    return result